# KASA-42 — H200 runbook

GPU window: **Thu 30 Jul 2026, 08:31 GMT → Sat 1 Aug, 08:31 GMT**

This notebook is a driver, not a codebase. Everything lives in `src/kasa42/`
so the kernel dying does not cost you any work. If a cell fails, fix the `.py`
file, re-run the import cell, and continue.

Order matters. Do not skip the smoke run to save 20 minutes.

## H+0 · Land

First: is the data already on this box? Ask before downloading 222 GB.

In [ ]:
!nvidia-smi
!df -h . | tail -2
# Look for a pre-staged copy before pulling anything.
!ls -la ~/.cache/huggingface/datasets 2>/dev/null | head
!find / -maxdepth 4 -iname '*ghana*speech*' -not -path '*/proc/*' 2>/dev/null | head

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

!git clone https://github.com/NasamuAlhassan/kasa42.git || (cd kasa42 && git pull --ff-only)
%cd kasa42
!pip install -q -e '.[train,serve]'

# `!python -m kasa42...` runs in a subprocess, where the kernel's sys.path does
# not apply. The editable install above handles it; PYTHONPATH is the backup.
os.environ['PYTHONPATH'] = os.path.join(os.getcwd(), 'src')
print(subprocess.run([sys.executable, '-c', 'import kasa42; print("subprocess import ok")'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import sys, torch
sys.path.insert(0, 'src')
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

### Data

`results/manifest.parquet`, `splits.json`, `vocab.json` and `mixture.json` were
built before the window and are committed. Rebuild only if something looks off.

Audio still has to land. Per-language caps mean you do **not** need all 222 GB —
download the configs the mixture actually draws from.

In [ ]:
import json
mix = json.load(open('results/mixture.json'))
want = sorted(c for c, ids in mix['segment_ids'].items() if ids)
print(len(want), 'configs needed')

from huggingface_hub import snapshot_download
snapshot_download('ghananlpcommunity/ghana-speech', repo_type='dataset',
                  local_dir='data/parquet',
                  allow_patterns=[f'{c}/*' for c in want],
                  max_workers=16)

## H+1 · Smoke run

30 steps on 5 languages. Proves the loop before you commit hours to it.

In [ ]:
from kasa42.asr.train import train, TrainConfig
train(TrainConfig(smoke=True,
                  languages=['Kusaal_kus','Asante_Twi_twi','Ewe_ewe','Dagaare_dga','Mampruli_maw'],
                  out_dir='checkpoints/smoke'))

## H+2 · Baselines first

Do this before the long run. If training disappoints you still have a story;
if it succeeds the numbers are already on the same axis.

In [ ]:
!python -m kasa42.asr.baselines --which dondo mms whisper --out-dir results/baselines

## H+4 · Main ASR run (~4–7 h)

In [ ]:
train(TrainConfig(max_steps=12000, batch_duration=320.0,
                  out_dir='checkpoints/kasa42-asr'))

## H+11 · Kusaal TTS

**Listen to `data/tts/check/*.wav` before starting the fine-tune.** If those
clips are not the same voice, stop and spend the hours on ASR instead.

In [ ]:
!python -m kasa42.tts.prepare --config Kusaal_kus
import IPython.display as ipd, glob
for f in sorted(glob.glob('data/tts/check/*.wav')):
    print(f); ipd.display(ipd.Audio(f))

In [ ]:
!python -m kasa42.tts.finetune --epochs 60

## H+15 · Evaluate — the leaked-vs-honest table is the headline

In [ ]:
!python -m kasa42.tts.roundtrip
# Per-language WER/CER + the leak comparison; see asr/evaluate.py

## H+23 · Export — do not leave this to the end

The GPU disappears Sat 08:31 GMT. If judging is after that, this cell **is**
the submission.

In [ ]:
!python -m kasa42.asr.export --checkpoint checkpoints/kasa42-asr/final.pt
# Then verify with the GPU hidden — this is the check that matters:
!CUDA_VISIBLE_DEVICES='' KASA42_MODE=onnx python -c "\
import sys; sys.path.insert(0,'src'); from kasa42.app.app import Engine; \
import numpy as np, time; e=Engine('onnx'); \
t,l,c,dt=e.transcribe(16000, np.zeros(16000,dtype=np.float32)); \
print('mode', e.mode, '|', l, f'{dt*1000:.0f}ms')"

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
# api.upload_folder(folder_path='export', repo_id='PrinceAlhassanNasamu/kasa42-asr', repo_type='model')
# api.upload_folder(folder_path='app',    repo_id='PrinceAlhassanNasamu/kasa42',     repo_type='space')